## Business Problem: Measuring Digital Advertising Effectiveness
#### A digital marketing company is investing in commercial advertisements to acquire customers and generate conversions. However, it is uncertain whether displaying commercial advertisements actually drives more conversions than showing neutral public service announcements (PSAs).

#### The company needs to evaluate whether its advertising campaigns generate incremental conversions and whether continuing to invest in commercial ad exposure is justified.

## Project Objective
#### Use A/B testing and statistical inference to determine whether commercial advertising produces a measurable improvement in conversions and provide evidence-based recommendations for marketing campaign decisions.

In [1]:
import pandas as pd
import numpy as np
import statsmodels as sm
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv("marketing_AB.csv")

Quick summary of the 7 columns from the dataset:

1. user id: Unique ID assigned to each user.
2. test group: Which experiment group the user was in (ad for advertisement, psa for public service announcement).
3. converted: Whether the user bought the product.
4. total ads: Total number of ads the person saw.
5. most ads day: The day of the week the person saw the most ads.
6. most ads hour: The hour of the day (0–23) when they saw the most ads.

In [3]:
df.head()

,user id,test group,converted,total ads,most ads day,most ads hour
0,1069124,ad,False,130,Monday,20
1,1119715,ad,False,93,Tuesday,22
2,1144181,ad,False,21,Tuesday,18
3,1435133,ad,False,355,Tuesday,10
4,1015700,ad,False,276,Friday,14


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 588101 entries, 0 to 588100
Data columns (total 6 columns):
 #   Column         Non-Null Count   Dtype
---  ------         --------------   -----
 0   user id        588101 non-null  int64
 1   test group     588101 non-null  str  
 2   converted      588101 non-null  bool 
 3   total ads      588101 non-null  int64
 4   most ads day   588101 non-null  str  
 5   most ads hour  588101 non-null  int64
dtypes: bool(1), int64(3), str(2)
memory usage: 23.0 MB


In [5]:
df.columns = df.columns.str.replace(' ','_')
df.columns

Index(['user_id', 'test_group', 'converted', 'total_ads', 'most_ads_day',
       'most_ads_hour'],
      dtype='str')

In [6]:
df = df.rename(columns = {'converted':'is_converted'})
df.head()

,user_id,test_group,is_converted,total_ads,most_ads_day,most_ads_hour
0,1069124,ad,False,130,Monday,20
1,1119715,ad,False,93,Tuesday,22
2,1144181,ad,False,21,Tuesday,18
3,1435133,ad,False,355,Tuesday,10
4,1015700,ad,False,276,Friday,14


In [7]:
#checking duplicates
df.duplicated().sum()

np.int64(0)

In [8]:
df.groupby('user_id')['user_id'].count().unique()

array([1])

In [9]:
df.describe(include ='all')

,user_id,test_group,is_converted,total_ads,most_ads_day,most_ads_hour
count,5.881010e+05,588101,588101,588101.000000,588101,588101.000000
unique,NaN,2,2,NaN,7,NaN
top,NaN,ad,False,NaN,Friday,NaN
freq,NaN,564577,573258,NaN,92608,NaN
mean,1.310692e+06,NaN,NaN,24.820876,NaN,14.469061
std,2.022260e+05,NaN,NaN,43.715181,NaN,4.834634
min,9.000000e+05,NaN,NaN,1.000000,NaN,0.000000
25%,1.143190e+06,NaN,NaN,4.000000,NaN,11.000000
50%,1.313725e+06,NaN,NaN,13.000000,NaN,14.000000
75%,1.484088e+06,NaN,NaN,27.000000,NaN,18.000000


In [10]:
#checking unique values in categorical column

for col in ['test_group','is_converted','most_ads_day','most_ads_hour']:
    print('unique values in',col,':',df[col].unique())
    print('='*50)

unique values in test_group : <StringArray>
['ad', 'psa']
Length: 2, dtype: str
unique values in is_converted : [False  True]
unique values in most_ads_day : <StringArray>
['Monday', 'Tuesday', 'Friday', 'Saturday', 'Wednesday', 'Sunday', 'Thursday']
Length: 7, dtype: str
unique values in most_ads_hour : [20 22 18 10 14 13 19 11 12 16 21  3 23  4  8  0  2 15  1  6 17  7  9  5]


In [11]:
df['is_converted'] = df['is_converted'].astype('int64')
              #.str.replace({False:0 , True: 1})

In [12]:
df['is_converted'].value_counts()

is_converted
0    573258
1     14843
Name: count, dtype: int64

In [13]:
summary = df.groupby('test_group').agg(
    total_users = ('user_id','count'),
    total_conversion = ('is_converted','sum'),
    conversion_rate = ('is_converted','mean'),
    avg_ads_seen = ('total_ads','mean')).reset_index()
summary

,test_group,total_users,total_conversion,conversion_rate,avg_ads_seen
0,ad,564577,14423,0.025547,24.823365
1,psa,23524,420,0.017854,24.761138


In [14]:
# Making a time bucket for different times of the day

def time_mapping(hour):
    if hour <= 5:
        return 'morning'
    elif hour <= 11:
        return 'afternoon'
    elif hour <= 17:
        return 'evening'
    else:
        return 'night'


    

In [15]:
df['day_time'] = df['most_ads_hour'].map(time_mapping)
df.head()

,user_id,test_group,is_converted,total_ads,most_ads_day,most_ads_hour,day_time
0,1069124,ad,0,130,Monday,20,night
1,1119715,ad,0,93,Tuesday,22,night
2,1144181,ad,0,21,Tuesday,18,night
3,1435133,ad,0,355,Tuesday,10,afternoon
4,1015700,ad,0,276,Friday,14,evening


## A/B Testing Experiment 
#### - Control Group (A): Users exposed to Public Service Announcements (PSAs).
#### - Treatment Group (B): Users exposed to commercial advertisements.

The experiment aims to determine whether exposure to commercial advertisements leads to a statistically significant improvement in user conversions compared with PSAs.

Experiment type: Two-arm controlled A/B test

Independent variable: Advertisement type (Commercial Ad vs PSA)

Dependent variable: User conversion (Binary: 0/1)

Primary KPI: Conversion Rate

Statistical approach: Two-proportion hypothesis test at a 5% significance level.

## Null & Alternate Hypothesis

Null Hypothesis, H0: Commercial ads have no or lower conversion rate than PSAs

Alternate Hypothesis, H1: Commercial ads have higher conversion rate than PSAs

In [16]:
# Performing two proportion z-test to see if there is difference between converion from two method of advertisements

from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep

In [17]:
z_stat, p_value = proportions_ztest(
                                    count = summary['total_conversion'],
                                    nobs = summary['total_users'],
                                    alternative = 'larger'
                                    )

z_stat, p_value

(np.float64(7.3700781265454145), np.float64(8.526403580779863e-14))

#### Now that z test has established that there is a gap but by how much? Lets check confidence interval

In [18]:
ad_conv = summary['total_conversion'][0]
ad_users = summary['total_users'][0]
psa_conv = summary['total_conversion'][1]
psa_users = summary['total_users'][1]

In [19]:

low, high = confint_proportions_2indep(ad_conv, ad_users, psa_conv, psa_users, compare = 'diff')
low, high

(np.float64(0.005873422353060636), np.float64(0.009360028811460754))

#### The ad group converted at a significantly higher rate than the PSA group (z = 7.37, p < 0.001). The estimated absolute lift is about 0.76 percentage points (95% CI: 0.59 to 0.94 pp). We reject the null hypothesis that the ad has no effect on conversion.

In [20]:
# Checking absolute and relative lift

absolute_lift = summary['conversion_rate'][0] - summary['conversion_rate'][1]
relative_lift = absolute_lift/summary['conversion_rate'][1]

print('Absolute lift : ', round(absolute_lift*100,2) ,'percentage points')
print('Relative lift : ', round(relative_lift*100,2) ,'percentage points')

Absolute lift :  0.77 percentage points
Relative lift :  43.09 percentage points


#### **POWER ANALYSIS**: Now I am doing power analysis in order to check if my data was sufficient to catch a meaningful interpretation or is there any risk of type I error or type II error.

In [21]:
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

In [22]:
es = proportion_effectsize(summary['conversion_rate'][0],summary['conversion_rate'][1])

In [23]:
power = NormalIndPower().power(effect_size = es, nobs1 = psa_users, ratio = ad_users/psa_users, alpha = 0.05)
print(power)

0.9999999990438161


#### Since power came out 0.9999, it means that the z-test would almost certainly (99.99% of the time) detect any real effect.  In short, your sample was large enough that missing a real effect was practically impossible.

In [24]:
mde_es = NormalIndPower().solve_power(nobs1 = psa_users, ratio = ad_users/psa_users, alpha = 0.05, power = 0.8, alternative = 'larger')
print(mde_es)

0.016545975380642872


In [25]:
import numpy as np

phi_psa = 2 * np.arcsin(np.sqrt(psa_conv/psa_users))
p_ad_mde = np.sin((phi_psa + mde_es) / 2) ** 2

mde_abs = p_ad_mde - psa_conv/psa_users
mde_rel = mde_abs / (psa_conv/psa_users)

print(f'Absolute mde :   {mde_abs*100 :.2f} pp')
print(f'Relative mde :   {mde_rel*100 : .2f} %')

Absolute mde :   0.23 pp
Relative mde :    12.64 %


#### The test could detect lifts as small as 0.23 pp (12.64% relative), and the observed lift of about 0.76 pp is well above that, so the test was sensitive enough for this result.

#### Running regression to check if the ad still help after we account for other things, like the day, the time of day, and how many ads people saw?

In [26]:
import statsmodels.formula.api as smf

In [27]:
# Regression model
model = smf.logit("is_converted ~ C(test_group, Treatment('psa')) + total_ads + C(day_time) + C(most_ads_day) ", data = df).fit() 

Optimization terminated successfully.
         Current function value: 0.108269
         Iterations 8


In [28]:
print(model.summary())

                           Logit Regression Results                           
Dep. Variable:           is_converted   No. Observations:               588101
Model:                          Logit   Df Residuals:                   588089
Method:                           MLE   Df Model:                           11
Date:                Mon, 16 Nov 2026   Pseudo R-squ.:                 0.08076
Time:                        17:54:23   Log-Likelihood:                -63673.
converged:                       True   LL-Null:                       -69267.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                            coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------------------
Intercept                                -4.8822      0.059    -83.404      0.000      -4.997      -4.767
C(test_group, Treatment('psa'))[T.ad]     0.3853  

In [29]:
OR = np.exp(0.3853)
OR.round(2)

np.float64(1.47)

In [30]:
CI_low = np.exp(0.284)
CI_high = np.exp(0.486)

print(f'CI_low: {CI_low}')
print(f'CI_high: {CI_high}')

CI_low: 1.3284329307527947
CI_high: 1.6257999962113414


#### After controlling for day of week, time of day and ad exposure, users shown the ad had about 47% higher odds of converting than the PSA group (OR ≈ 1.47, 95% CI: 1.33 to 1.63, p < 0.001). This is consistent with the z-test, so the lift isn’t explained by differences in when users saw ads. Conversion was also higher in the evening and at night, lower in the morning, and higher early in the week.

In [31]:
# model without total_ads
model_wo_ads = smf.logit("is_converted ~ C(test_group, Treatment('psa')) + C(day_time) + C(most_ads_day) ", data = df).fit() 

Optimization terminated successfully.
         Current function value: 0.117102
         Iterations 8


In [32]:
print(model_wo_ads.summary())

                           Logit Regression Results                           
Dep. Variable:           is_converted   No. Observations:               588101
Model:                          Logit   Df Residuals:                   588090
Method:                           MLE   Df Model:                           10
Date:                Mon, 16 Nov 2026   Pseudo R-squ.:                0.005763
Time:                        17:54:37   Log-Likelihood:                -68868.
converged:                       True   LL-Null:                       -69267.
Covariance Type:            nonrobust   LLR p-value:                4.648e-165
                                            coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------------------
Intercept                                -4.3427      0.056    -77.536      0.000      -4.452      -4.233
C(test_group, Treatment('psa'))[T.ad]     0.3653  

In [35]:
OR = np.exp(0.3653)
CI_low = np.exp(0.267)
CI_high = np.exp(0.463)

print(f"OR : {OR}")
print(f"CI Low = {CI_low}")
print(f"CI High = {CI_high}")

OR : 1.440946227181275
CI Low = 1.3060404463306543
CI High = 1.58883334241608


In [34]:
# 1. Upload refined data Table
df.to_csv("refined_data.csv")

# 2. Upload summary Table
summary.to_csv("summary.csv")


print("Both the tables (data, summary) uploaded successfully to 'marketing_db'!")

Both the tables (data, summary) uploaded successfully to 'marketing_db'!
